# Analisis Statistik — Sistem Dummy Coretax
Notebook ini mereproduksi angka inferensial artikel (H1–H14) dari data respons individual (anonim, N = 96).

**Cara pakai:** *Runtime → Run all*. Jika repositori sudah di GitHub, isi `REPO_URL` pada sel 1; jika belum, sel 1 akan meminta Anda mengunggah `data_respons_individual.csv`, `analisis_statistik.py`, dan `buat_gambar.py`.

In [ ]:
import os, subprocess
REPO_URL = ''   # contoh: 'https://github.com/USERNAME/coretax-dummy-tax-education.git'
if REPO_URL:
    if not os.path.exists('repo'): subprocess.run(['git','clone',REPO_URL,'repo'],check=True)
elif not os.path.exists('repo/data/data_respons_individual.csv'):
    from google.colab import files   # unggah 3 berkas
    os.makedirs('repo/data',exist_ok=True); os.makedirs('repo/code',exist_ok=True)
    for n in files.upload():
        os.replace(n, ('repo/data/' if n.endswith('.csv') else 'repo/code/')+n)
ROOT = 'repo'
print('ROOT =', ROOT)

In [ ]:
!pip -q install numpy pandas scipy matplotlib

## 1. Pratinjau data

In [ ]:
import pandas as pd
df = pd.read_csv(f'{ROOT}/data/data_respons_individual.csv')
print(df.shape, '| nilai kosong:', int(df.isna().sum().sum()))
df.head()

## 2. Jalankan analisis (bootstrap 5.000, ± 1–3 menit)
Untuk uji cepat ubah `--boot 5000` menjadi `--boot 500`.

In [ ]:
!python {ROOT}/code/analisis_statistik.py --boot 5000

## 3. Ringkasan hasil utama

In [ ]:
import json
R = json.load(open(f'{ROOT}/results/hasil_analisis.json'))
H1=R['H1']; S=R['SUS']
print(f"H1  post {H1['post_m']:.2f} vs pre {H1['pre_m']:.2f}; t(95)={H1['t']:.2f}; dz={H1['dz']:.2f}; N-Gain={H1['ng_m']:.2f}")
print(f"H2  SUS {S['m']:.2f} (SD {S['sd']:.2f}); Wilcoxon z={S['z']:.2f}; d={S['d']:.2f}")
print(f"H3a PU {R['PU']['m']:.2f}; H3b PEOU {R['PEOU']['m']:.2f}")
print('\nJalur struktural (beta, CI 95%):')
for k,v in R['boot']['B'].items(): print(f"  {k:10s} {v['est']:.3f}  [{v['lo']:.3f}; {v['hi']:.3f}]")
print('\nEfek tidak langsung:')
for k,v in R['boot']['I'].items(): print(f"  {k:22s} {v['est']:.3f}  [{v['lo']:.3f}; {v['hi']:.3f}]")
m=R['boot']['M']['inter']; print(f"\nH14 interaksi TCSE x PEOU: {m['est']:.3f}  [{m['lo']:.3f}; {m['hi']:.3f}]")

## 4. Gambar 2 dan 3

In [ ]:
!python {ROOT}/code/buat_gambar.py
from IPython.display import Image, display
display(Image(f'{ROOT}/figures/gambar2_model_tax_tam.png')); display(Image(f'{ROOT}/figures/gambar3_simple_slopes.png'))